# MCP Concepts 03: How an Agent "Picks" a Server With Multiple MCP Servers

## Problem card

- **The question this notebook actually answers:** when one LangGraph agent
  is connected to *three* MCP servers at once, how does it decide which
  *server* to use? The honest answer, proven below, is that **it doesn't** --
  there is no server-level decision at all. `MultiServerMCPClient.get_tools()`
  flattens every server's tools into one list bound to the LLM, and the model
  just does ordinary tool selection over that flat list, the same mechanism
  as any single-server agent. "Which server" is a side effect of "which
  tool," never a decision on its own.
- **Why that's not just trivia:** it means multi-server MCP is NOT the same
  pattern as `multi_agent_architectures/01_supervisor_orchestrator_worker.ipynb`,
  where a supervisor explicitly routes to one of several domains. There is no
  supervisor here -- and that has a real, demonstrable consequence when two
  servers happen to expose a tool with the same name, which this notebook
  reproduces for real, not hypothetically.
- **Servers used:** `calculator_server.py`, `knowledge_ops_server.py`, and
  `orders_server.py` -- all connected to the same agent simultaneously over
  stdio.

## Architecture

```mermaid
flowchart TD
    U[User question] --> A["agent node\n(LLM sees ONE flat tool list)"]
    A --> T[tools node]
    T -.-> CS[calculator_server.py]
    T -.-> KS[knowledge_ops_server.py]
    T -.-> OS[orders_server.py]

    style A fill:#a8dadc,stroke:#333
    style T fill:#f9c74f,stroke:#333
```

Compare this to a supervisor pattern, where a dedicated routing node picks
**one destination** before any tool is considered. Here, there is no such
node -- the flattening happens once, up front, in `get_tools()`.

In [1]:
import os
import sys
import warnings
from typing import TypedDict, Annotated
from dotenv import load_dotenv, find_dotenv

warnings.filterwarnings("ignore")
load_dotenv(find_dotenv(usecwd=True))

sys.path.insert(0, os.getcwd())
import shared

from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_openai import ChatOpenAI

scorecard = shared.ScorecardCallback()
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"), temperature=0, callbacks=[scorecard])

SERVERS = {
    "calculator": {"command": "python3", "args": [os.path.join(os.getcwd(), "servers", "calculator_server.py")], "transport": "stdio"},
    "knowledge_ops": {"command": "python3", "args": [os.path.join(os.getcwd(), "servers", "knowledge_ops_server.py")], "transport": "stdio"},
    "orders": {"command": "python3", "args": [os.path.join(os.getcwd(), "servers", "orders_server.py")], "transport": "stdio"},
}
mcp_client = MultiServerMCPClient(SERVERS)
all_tools = await mcp_client.get_tools()

print(f"Connected to {len(SERVERS)} servers, {len(all_tools)} tools total (flat list):")
names = [t.name for t in all_tools]
for n in names:
    marker = "  <-- DUPLICATE NAME" if names.count(n) > 1 else ""
    print(f"  - {n}{marker}")

Connected to 3 servers, 11 tools total (flat list):
  - add
  - subtract
  - multiply
  - divide
  - percentage_of
  - search_runbooks
  - get_service_status
  - check_status  <-- DUPLICATE NAME
  - lookup_order_status
  - get_shipping_estimate
  - check_status  <-- DUPLICATE NAME


Notice `check_status` appears **twice** in that list -- once from
`knowledge_ops_server.py`, once from `orders_server.py`. `get_tools()`
doesn't namespace tool names by server, so nothing here prevents two
different servers from colliding. Keep this in mind; the second half of this
notebook proves it's a real problem, not a cosmetic one.

## Part 1: normal cross-server routing (the common case, working correctly)

Three questions, each needing a *different* server's tool. Watch which tool
gets called for each -- this is ordinary LLM tool-selection, nothing
server-aware about it.

In [2]:
async def run_agent(tools, question: str):
    class State(TypedDict):
        messages: Annotated[list, add_messages]

    llm_with_tools = llm.bind_tools(tools)

    async def agent_node(state):
        resp = await llm_with_tools.ainvoke(state["messages"])
        return {"messages": [resp]}

    builder = StateGraph(State)
    builder.add_node("agent", agent_node)
    builder.add_node("tools", ToolNode(tools))
    builder.add_edge(START, "agent")
    builder.add_conditional_edges("agent", tools_condition, {"tools": "tools", "__end__": END})
    builder.add_edge("tools", "agent")
    graph = builder.compile()

    result = await graph.ainvoke({"messages": [{"role": "user", "content": question}]})
    calls = [tc["name"] for m in result["messages"] for tc in (getattr(m, "tool_calls", None) or [])]
    final = result["messages"][-1].content
    return calls, final

In [3]:
questions = [
    "What's 12 multiplied by 7?",
    "What's the shipping ETA for order ORD-1001?",
    "Search the runbooks for anything about 'search'.",
]

for q in questions:
    calls, final = await run_agent(all_tools, q)
    print(f"Q: {q}")
    print(f"   tool(s) called: {calls}")
    print(f"   answer: {final}\n")

Q: What's 12 multiplied by 7?
   tool(s) called: ['multiply']
   answer: 12 multiplied by 7 is 84.



Q: What's the shipping ETA for order ORD-1001?
   tool(s) called: ['get_shipping_estimate']
   answer: The shipping ETA for order ORD-1001 is 2 days, and the carrier is FastShip.



Q: Search the runbooks for anything about 'search'.
   tool(s) called: ['search_runbooks']
   answer: I found a runbook titled "search." If you need more details about it, please let me know!



Each question correctly landed on the one server whose tool actually
matched -- `multiply` (calculator), `get_shipping_estimate` (orders),
`search_runbooks` (knowledge_ops). No routing logic decided this; the tool
*names and descriptions* did all the work, exactly like a single-server
agent.

## Part 2: the collision, reproduced for real

Bind *only* the two `check_status` tools (one per colliding server) and ask
about a **service** -- something only the `knowledge_ops` version of
`check_status` could actually answer.

In [4]:
check_status_tools = [t for t in all_tools if t.name == "check_status"]
print(f"Tools literally named 'check_status': {len(check_status_tools)}")
for t in check_status_tools:
    origin = "knowledge_ops_server.py" if "orders_server.py's module" in t.description else "orders_server.py"
    print(f"  - from {origin}: {t.description.splitlines()[0]}")

node = ToolNode(check_status_tools)
survivor = node.tools_by_name["check_status"]
survivor_origin = "knowledge_ops_server.py" if "orders_server.py's module" in survivor.description else "orders_server.py"
print(f"\nToolNode.tools_by_name has only ONE 'check_status' entry -- the one from {survivor_origin}.")
print("The other server's check_status is now unreachable through this ToolNode, no matter what the model intends.")

Tools literally named 'check_status': 2
  - from knowledge_ops_server.py: Check status by id. (Deliberately vague -- see orders_server.py's module
  - from orders_server.py: Check status by id. (Deliberately vague -- see module docstring; used only

ToolNode.tools_by_name has only ONE 'check_status' entry -- the one from orders_server.py.
The other server's check_status is now unreachable through this ToolNode, no matter what the model intends.


In [5]:
calls, final = await run_agent(check_status_tools, "What's the status of the search service?")
print(f"tool(s) called: {calls}")
print(f"answer: {final}")

tool(s) called: ['check_status']
answer: It seems that I cannot directly check the status of the search service as it is not recognized. However, I can check the status of specific orders. If you have an order ID related to the search service, please provide it, and I can check the status for you. The known order IDs are: ORD-1001, ORD-1002, and ORD-1042.


**What actually happened:** the model correctly decided `check_status` was
the right tool and passed a sensible id (`search_service` or similar) --
its reasoning wasn't the problem. The problem is that
`ToolNode.tools_by_name` is a plain dict keyed by name, so when two MCP
servers both register `check_status`, **only the last one registered
survives** -- the other becomes silently unreachable. Here that means every
`check_status` call, no matter what the model intended, is actually
executed by the `orders` server's version, which has no idea what a
"service" is and returns an error. This is a **deterministic consequence of
duplicate tool names**, not an LLM mistake -- rerunning this cell will
reproduce the same failure every time, regardless of model or temperature.

## Part 3: the fix

Don't rely on generic, server-agnostic tool names across multiple MCP
servers you're combining. `get_service_status` and `lookup_order_status`
are unambiguous and don't collide -- re-running the same question with the
well-named tools available fixes it immediately.

In [6]:
well_named_tools = [t for t in all_tools if t.name in ("get_service_status", "lookup_order_status")]
calls, final = await run_agent(well_named_tools, "What's the status of the search service?")
print(f"tool(s) called: {calls}")
print(f"answer: {final}")

tool(s) called: ['get_service_status']
answer: The status of the search service is currently degraded, with an error rate of 0.3%. The last deployment occurred 25 minutes ago.


## Takeaway

| | Supervisor pattern (notebook 01, multi_agent_architectures) | Multi-server MCP (this notebook) |
|---|---|---|
| Who picks the destination | An explicit routing node/LLM call | Nobody -- there is no destination concept, only a flat tool list |
| What "wrong choice" looks like | Wrong domain picked, tools from the right domain never considered | Right tool name picked, but a **name collision** silently redirects it to the wrong server's implementation |
| The fix | Better routing prompt/examples | Unique, well-scoped tool names across every server you combine (or explicit namespacing if you don't control server code) |

If you're integrating MCP servers you don't control (third-party or public
servers -- see notebook 04), you cannot assume their tool names won't
collide with each other or with your own. Treat that as an integration risk
to check for explicitly, not an edge case.

## Explain like I'm 12

Imagine two different food trucks parked next to each other, and both
happen to have a button on their counter labeled just "STATUS" -- one
truck's button tells you if *your pizza order* is ready, the other tells
you if *the truck itself* is open or closed. If someone wires both buttons
to ring the exact same bell, and only one truck's wire actually gets
connected, then no matter which truck you *meant* to ask, pressing "STATUS"
always rings the connected truck's bell -- even if you were asking about the
other truck entirely. The fix isn't to train yourself to guess better; it's
to put clearer, different labels on the buttons ("ORDER STATUS" vs. "TRUCK
STATUS") so there's only ever one truck a given button could mean.

## Checkpoint questions

1. **Q: Does `MultiServerMCPClient` warn you when two servers register a
   tool with the same name?**
   A: No -- `get_tools()` returns them both in the flat list without
   complaint; the collision only becomes visible once you inspect
   `ToolNode.tools_by_name` (or hit the wrong behavior at runtime, as this
   notebook did).

2. **Q: In the collision demo, was the LLM's tool-call decision actually
   wrong?**
   A: No -- it correctly chose `check_status` and passed a reasonable
   argument. The failure happened entirely inside `ToolNode`'s
   name-to-tool lookup, after the model had already made a sound decision.

3. **Q: If you don't control the source code of two MCP servers you want to
   combine, and they happen to collide on a tool name, what are your
   options?**
   A: Check whether the client library supports per-server tool-name
   prefixing/namespacing before combining them; if not, filter/rename tools
   client-side after `get_tools()`, or avoid binding the colliding pair
   together and route to them via separate agent instances instead.